# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\diana\Desktop\18.Ingineria_AI\echochamber-project-team3
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_05"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [6]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().iloc[:15]

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [7]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=36)

,source_channel,video_title,text
18025,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,Mi-aduc aminte reportajul cu ND prima zi la pr...
24302,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...",Nu sunt nici socialiști intrucat tot greul mer...
27928,turcescu111,Iran contra-atacă. Orientul Mijlociu în flăcări,Ce democraatie? Unde mai exista democrație? De...
3953,digi24hd56,Hectare întregi din pădurea Hoia-Baciu de lâng...,fenomene paranormale imobiliare. si apoi fenom...
16940,RecorderRomania,PORTRET DE CANDIDAT: George Simion,Simion e un personaj machiavelic. Ce obtii sem...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [8]:
sample_df = df.sample(10, random_state=36).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
18025,RecorderRomania,Mi-aduc aminte reportajul cu ND prima zi la pr...
24302,turcescu111,Nu sunt nici socialiști intrucat tot greul mer...
27928,turcescu111,Ce democraatie? Unde mai exista democrație? De...
3953,digi24hd56,fenomene paranormale imobiliare. si apoi fenom...
16940,RecorderRomania,Simion e un personaj machiavelic. Ce obtii sem...
24371,turcescu111,"Domnule Robert , Trump i-a dat un termen in ca..."
21869,CălinGeorgescu-CanalulOficial,Cel Mai IUBIT OM DIN ROMANIA ❤❤❤❤SI IN EUROPA ...
5825,@CălinGeorgescu-CanalulOficial,Despre Eminescu nu se poate vorbi decât cu ini...
5994,@CălinGeorgescu-CanalulOficial,"mai du-te si invarte-te odata, nene, ca ti-ai ..."
20777,CălinGeorgescu-CanalulOficial,❤️ Prezența dumneavoastră ne dă speranță și pu...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [9]:
SYSTEM_PROMPT = """
Ești un cercetător și analist politic expert, specializat în analiza discursului online. 
Scopul tău este să analizezi comentarii publice de pe YouTube pentru a detecta atitudini, emoții și tipare retorice. 
Trebuie să fii obiectiv, să analizezi strict textul oferit și să returnezi rezultatul EXCLUSIV într-un format de date JSON.
"""

USER_PROMPT_TEMPLATE = """
Analizează următorul comentariu politic și extrage următoarele elemente:

1. target: (string) Entitatea principală vizată de comentariu (ex: un politician anume, o instituție, 'clasa politică', 'mass-media', sau 'nespecificat').
2. stance: (string) Poziția autorului față de țintă ("PRO", "CONTRA", "NEUTRU")
3. sentiment: (string) Valența emoțională generală a textului. Alege strict dintre: 'pozitiv', 'negativ', 'neutru'.
4. tone: (string) Tonul predominant al discursului (ex: 'agresiv', 'sarcastic', 'indignat', 'resemnat', 'informativ', 'victimizare', 'rezoltat', 'defensiv').
5. topic: (string) Subiectul sau tema principală discutată (ex: 'corupție', 'justiție', 'alegeri', 'economie', 'suveranitate').
6. interpretation_problem: (string) Notează dacă există dificultăți în analiza textului (ex: 'sarcasm puternic', 'incoerent', 'lipsă de context'). Dacă comentariul e clar, scrie 'niciuna'.
7. justification: (string) scurtă justificare în română

Important:
- Bazează-te STRICT pe comentariul dat, nu inventa intenții care nu reies din text.
- Nu adăuga niciun fel de text explicativ înainte sau după JSON.

Returnează JSON valid cu exact aceste chei (observă acoladele duble obligatorii pentru codul Python):
{{
    "target": "...",
    "stance": "...",
    "sentiment": "...",
    "tone": "...",
    "topic": "...",
    "interpretation_problem": "...",
    "justification": "..."
}}

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [10]:
from openai import OpenAI
client = OpenAI(
    api_key= GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [11]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [12]:
import time
from openai import RateLimitError

n_comments = 10  
sample_for_prompt = sample_df.head(n_comments)

outputs = []
for index, row in sample_for_prompt.iterrows():
    print(f"Adnotez comentariul {len(outputs) + 1} din {n_comments}...")
    
    success = False
    while not success:
        try:
            # Încercăm să apelăm modelul
            rezultat = annotate_comment(row["text"])
            
            outputs.append({
                "source_channel": row.get("source_channel", ""),
                "video_title": row.get("video_title", ""),
                "comment_text": row["text"],
                "model_output": rezultat
            })
            
            success = True # Dacă a mers, ieșim din bucla while
            time.sleep(5)  # Pauză normală de 5 secunde între comentarii
            
        except RateLimitError:
            # Dacă Google ne blochează, așteptăm 30 de secunde și modelul va reîncerca automat!
            print("⏳ Am atins limita de API. Așteptăm 30 de secunde...")
            time.sleep(30)
            
        except Exception as e:
            # Dacă apare altă eroare ciudată, o afișăm și trecem mai departe ca să nu pierdem progresul
            print(f"Eroare neașteptată la acest comentariu: {e}")
            success = True 

results_df = pd.DataFrame(outputs)
print("\nGata! Am terminat de adnotat tot.")
results_df

Adnotez comentariul 1 din 10...
Adnotez comentariul 2 din 10...
Adnotez comentariul 3 din 10...
Adnotez comentariul 4 din 10...
Adnotez comentariul 5 din 10...
Adnotez comentariul 6 din 10...
Adnotez comentariul 7 din 10...
Adnotez comentariul 8 din 10...
Eroare neașteptată la acest comentariu: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]
Adnotez comentariul 8 din 10...
Adnotez comentariul 9 din 10...

Gata! Am terminat de adnotat tot.


,source_channel,video_title,comment_text,model_output
0,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,Mi-aduc aminte reportajul cu ND prima zi la pr...,"```json\n{\n ""target"": ""Nicolae Duduianu (N..."
1,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...",Nu sunt nici socialiști intrucat tot greul mer...,"```json\n{\n ""target"": ""Partidele din Parla..."
2,turcescu111,Iran contra-atacă. Orientul Mijlociu în flăcări,Ce democraatie? Unde mai exista democrație? De...,"```json\n{\n ""target"": ""democrație"",\n ""..."
3,digi24hd56,Hectare întregi din pădurea Hoia-Baciu de lâng...,fenomene paranormale imobiliare. si apoi fenom...,"```json\n{\n ""target"": ""clasa politică"",\n ..."
4,RecorderRomania,PORTRET DE CANDIDAT: George Simion,Simion e un personaj machiavelic. Ce obtii sem...,"```json\n{\n ""target"": ""George Simion"",\n ..."
5,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...","Domnule Robert , Trump i-a dat un termen in ca...","```json\n{\n ""target"": ""Trump"",\n ""stanc..."
6,CălinGeorgescu-CanalulOficial,Călin Georgescu - Și a plâns Iisus ( 21.10.202...,Cel Mai IUBIT OM DIN ROMANIA ❤❤❤❤SI IN EUROPA ...,"```json\n{\n ""target"": ""Călin Georgescu"",\n..."
7,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Nu suntem singuri! ( 13.01.2...,"mai du-te si invarte-te odata, nene, ca ti-ai ...","```json\n{\n ""target"": ""nespecificat"",\n ..."
8,CălinGeorgescu-CanalulOficial,"Călin Georgescu - SUVERANITATEA - la Buftea, o...",❤️ Prezența dumneavoastră ne dă speranță și pu...,"```json\n{\n ""target"": ""nespecificat"",\n ..."


# 9. Verificam rezultatele

In [13]:
results_df.model_output[0]

'```json\n{\n    "target": "Nicolae Duduianu (ND)",\n    "stance": "CONTRA",\n    "sentiment": "negativ",\n    "tone": "sarcastic",\n    "topic": "performanța administrativă",\n    "interpretation_problem": "niciuna",\n    "justification": "Autorul face o comparație sarcastică între așteptările inițiale legate de activitatea unui politician (ND) și lipsa de realizări concrete, sugerând o performanță slabă."\n}\n```'

In [14]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [15]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,Mi-aduc aminte reportajul cu ND prima zi la pr...,Nicolae Duduianu (ND),CONTRA,negativ,sarcastic,performanța administrativă,niciuna,
1,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...",Nu sunt nici socialiști intrucat tot greul mer...,Partidele din Parlamentul României si cel Euro...,CONTRA,negativ,indignat,economie,niciuna,
2,turcescu111,Iran contra-atacă. Orientul Mijlociu în flăcări,Ce democraatie? Unde mai exista democrație? De...,democrație,CONTRA,negativ,sarcastic,politică,niciuna,
3,digi24hd56,Hectare întregi din pădurea Hoia-Baciu de lâng...,fenomene paranormale imobiliare. si apoi fenom...,clasa politică,CONTRA,negativ,indignat,corupție,niciuna,
4,RecorderRomania,PORTRET DE CANDIDAT: George Simion,Simion e un personaj machiavelic. Ce obtii sem...,George Simion,CONTRA,negativ,agresiv,politică,niciuna,
5,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...","Domnule Robert , Trump i-a dat un termen in ca...",Trump,PRO,neutru,informativ,geopolitică,niciuna,
6,CălinGeorgescu-CanalulOficial,Călin Georgescu - Și a plâns Iisus ( 21.10.202...,Cel Mai IUBIT OM DIN ROMANIA ❤❤❤❤SI IN EUROPA ...,Călin Georgescu,PRO,pozitiv,admirativ,politică internă,niciuna,
7,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Nu suntem singuri! ( 13.01.2...,"mai du-te si invarte-te odata, nene, ca ti-ai ...",nespecificat,CONTRA,negativ,agresiv,nespecificat,niciuna,
8,CălinGeorgescu-CanalulOficial,"Călin Georgescu - SUVERANITATEA - la Buftea, o...",❤️ Prezența dumneavoastră ne dă speranță și pu...,nespecificat,PRO,pozitiv,admirativ,nespecificat,niciuna,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

Nu. Chiar de la primul comentariu acest model cade in plasa ND, identificand alt nume, Nicolae Duduianu, in loc de Nicusor Dan, nume prezent direct in titlul videoclipului. La comentariile 9 si 10, de asemenea, nu reuseste sa defineasca target-ul, Calin Georgescu, deoarece acesta nu este exprimat in mod direct, fiind prezent doar in titlul videoclipului. Modelul ar putea fi imbunatatit prin adaugarea de detalii la target, pentru a putea deduce contextul comentariului si din titlul videoclipului, pentru a evita tinte nespecificate. Se mai poate ajusta si timpul de call al API pentru a nu primi time-out.

In [16]:
results_df.to_csv(
    f"outputs/student_05_prompt_v1_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Salvat:", f"outputs/student_05_prompt_v1_review.csv")
print("Rânduri salvate:", len(results_df))
results_df.head()

Salvat: outputs/student_05_prompt_v1_review.csv
Rânduri salvate: 9


,source_channel,video_title,comment_text,model_output
0,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,Mi-aduc aminte reportajul cu ND prima zi la pr...,"```json\n{\n ""target"": ""Nicolae Duduianu (N..."
1,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...",Nu sunt nici socialiști intrucat tot greul mer...,"```json\n{\n ""target"": ""Partidele din Parla..."
2,turcescu111,Iran contra-atacă. Orientul Mijlociu în flăcări,Ce democraatie? Unde mai exista democrație? De...,"```json\n{\n ""target"": ""democrație"",\n ""..."
3,digi24hd56,Hectare întregi din pădurea Hoia-Baciu de lâng...,fenomene paranormale imobiliare. si apoi fenom...,"```json\n{\n ""target"": ""clasa politică"",\n ..."
4,RecorderRomania,PORTRET DE CANDIDAT: George Simion,Simion e un personaj machiavelic. Ce obtii sem...,"```json\n{\n ""target"": ""George Simion"",\n ..."
